### Optimizations can be performed at Spark , Delta Table and Databricks 

Spark
- Parititioning
- Bucketing
- Cache
- Persist
- Adaptive Query Execution(AQE) - Handling Skew Joins, BroadCast Join, Partition Pruning

Delta Table
- Z order
- Vaccuming
- Checkpoints
- Optimize
- Data Skipping

DataBricks
- Photon Engine
- Cluster Configuration
- Unity catalog
- Delta Cache
- AutoLoader

---
__Delta Table__

1. Delta Table stores data in **Parquet** format. It also creates transaction logs(metadata- it includes schema, changes, properties of table, version)

2. Delta table contains json and crc(checksum files). Json file contains commit information and metadata. Checksum File contains checksum values, it is used to validate partitions(other user not modified, data is not modified)

3. Each append creates one JSON and CRC files.
---

__Partitions__

Partition- Dividing file into multiple chunck, to process parallely

Two types of Partitions
1. InMermory Partition
2. Physical Partition

- configuration tirgger InMemory Configuration based on cluster(no. of parallel task) or spark config(partition size) 
  - spark.config.get("spark.sql.files.maxPartitionBytes")
- Default Partition size- 128MB
- To get no of partitions --> df.rdd.getNumPartitions()
- Partition By only works with write method --> df.write.partitionBy().save()
- To change the number of partition we can use repartition & coalesce
- in __repartition__ - partition size can be increased or decreased, it causes shuffling of data
- in **coalesce** - can be able to decrease the data, it does not cause full shuffling of data
---

__Caching__

Caching can be implemented from both Spark & DataBricks.
- Caching is used to imporve read operations or query retrival, used in complex computations.
###### Spark Caching
- caching can be implemented using cache() & persist()
- caching is explicit
- Spark caching is not optimized for delta tables
- if dataframe updated, rechaching is required
- Use unpersist() to remove cache

###### Delta Table Caching
- Both implicit(based on cluster type) and explicit(based on configuration)
- Recaching is not required
- spark.conf.set('spark.databricks.io.cache.enabled', True)
- if data caching is enabled , when you read the data from table it will be cached automatically and again if you try to read the data cached data is used to display the results 

###### Data Skipping
- Delta Table Feature
- Using metadata, minimum and maximum values of a column in JSON files, Databricks optimize the queries and only reads corresponding partitions and will skipps reading of all the partitions. This leads to the faster results and less data read

###### Optimize/ File Compaction/ Bin Packing
- Delta Table Feature
- *combine multiple small files into one big file*(default size ~1GB)
- Also retains small files, but delta engine reads big file
- Reason behind retaining small files:- To obtain versioning or TimeTravel
- Implemented using ```.optimize()```
- using ```.executeCompaction(targetsize='')``` default size of big file can be changed

###### Z ordering
- colocate similar type of data in a partition. 
- why zorder required: reading big partition file cotaining million of rows is time consuming
- cardinality = no of unique values
- Partitions are done on columns of low cardinality.
- Z Ordering are done on columns of high cardinality.
- ``` .optimize().executeZOrderBy(['column_name']) ```

###### Liquid Clustering

- *DrawBack of Z-Ordering / Partitioning* 
  1. No Flexiblility in Partitioning Column(if you change the joining key column- entire table needs repartitioning)
  2. Due to change in joining key - old physical partitions have to be deleted and re created
  3. During these optimizations, data cannot be written to table and re-optimization required

- In Liquid Clustering there will only **one partition** and *metadata will be stored for __all columns__*
- Z-Ordering and partitionBy is not allowed in Liquid Clustering
- 4 columns by default can be clusterd
- first 32 columns by default will be scanned
- Liquid Clustering is implemented by ```.clusterBy()```

